# 03 · 사용자 집계와 SRM

**핵심 실행 9분** · 좋아 보이는 지표를 해석하기 전에 데이터의 신뢰성을 확인합니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 종료된 합성 스냅샷 | healthy의 앞 5행과 authored_synthetic, 14일 종료 스냅샷 설명. |
| 3 | 집계 계약 검증 | 원본 검증은 [], 중복 행 추가 후 사용자 단위 중복 메시지. INVALID assert 통과. |
| 4 | 정상 배정과 비율 불일치 | healthy 1196/1204, p≈0.870. srm 1196/602, p≈1.38×10⁻⁴⁴. |
| 5 | 원래 배정과 수집 결과 대조 | B 602명 누락, INVALID, effect is None. |
| 6 | 성공 응답만 남기는 함정 | 전체 A 55.02% / B 64.04%, 성공 응답만 A 55.57% / B 64.95%. |
| 7 | 관찰 지표와 사람의 행동 | 전체 A 55.02% / B 64.04%, 피드백 응답자 내 A 62.02% / B 72.26%. |

**이번 파일의 역할:** 사용자 효과를 계산하기 전에 비교 집단과 분모를 검증합니다. 10주차 로그 존재 여부, 11주차 문항 정답 검증과 다른 데이터 신뢰성 문제입니다.

**재실행 방법:** 위에서 아래로 재실행 가능. duplicate·broken은 별도 객체이며 원본 CSV는 수정하지 않습니다. 셀 7은 셀 6의 all_users를 사용합니다.

**코드 셀 1 설명:** 실험을 비교하기 전에 실행 환경과 조건부터 맞춥니다. 파일별 커널은 독립적이므로 각 Notebook의 첫 코드 셀부터 실행합니다.

번호는 코드 셀 순서이며 실행 횟수 `[1]`, `[2]`와 다릅니다. [셀별 교수 설명](https://smilesjcha.github.io/llmops-16week/week12/lecture/instructor/#book-03)

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 종료된 합성 스냅샷

가상 14일 실험의 사용자별 첫 요청 결과입니다. 합성 데이터 생성 규칙은 공개 코드에 있습니다.

In [ ]:
data = load_units("healthy")
display(data.head())
metadata = json.loads((ROOT/"week12/lab/data/metadata.json").read_text())
print(metadata["notes"])

**확인과 해석:** healthy의 앞 5행과 authored_synthetic, 14일 종료 스냅샷 설명.

**화면에서 볼 곳:** visitor_id·variant·feedback_received·source

**설명 핵심:** 한 행은 첫 실험 진입 사용자 한 명입니다. 합성 데이터로 계산 방법을 배우는 것이며 실제 고객 효과를 주장할 데이터는 아닙니다.

**직접 설명:** 이 데이터로 실제 제품 효과를 주장하면 왜 잘못인가요?

## 코드 셀 3 · 집계 계약 검증

결측·중복·잘못된 버전은 계산 전에 막습니다. 무조건 dropna로 지우지 않습니다.

In [ ]:
from week12.lab.analysis import validate_units
assert validate_units(data) == []
duplicate = pd.concat([data,data.iloc[[0]]],ignore_index=True)
print(validate_units(duplicate))
assert analyze(duplicate)["decision"] == "INVALID"

**확인과 해석:** 원본 검증은 [], 중복 행 추가 후 사용자 단위 중복 메시지. INVALID assert 통과.

**화면에서 볼 곳:** validate_units와 duplicate

**설명 핵심:** 먼저 사람 단위를 정하고 중복을 검출합니다. 임의로 중복을 제거해 숫자를 맞추기보다 왜 같은 사람이 두 번 들어왔는지 원인을 찾습니다.

**직접 설명:** 반복 요청을 사용자가 늘어난 것으로 세면 신뢰구간에 어떤 문제가 생기나요?

## 코드 셀 4 · 정상 배정과 비율 불일치

SRM은 Sample Ratio Mismatch, 설정한 배정 비율과 관찰한 사용자 비율의 불일치입니다. 원인을 알려주는 판결이 아니라 진단 신호입니다.

In [ ]:
from week12.lab.analysis import srm_test
for name in ["healthy","srm"]:
    frame = load_units(name)
    n = frame.variant.value_counts()
    result = srm_test(int(n["A"]),int(n["B"]))
    print(name,n.to_dict(),result)

**확인과 해석:** healthy 1196/1204, p≈0.870. srm 1196/602, p≈1.38×10⁻⁴⁴.

**화면에서 볼 곳:** SRM: Sample Ratio Mismatch · 표본 비율 불일치

**설명 핵심:** SRM 검사는 누가 더 좋은 답변을 받았는지 판단하지 않습니다. 기대 배정 비율과 관측 인원이 맞는지를 확인하며, 강한 불일치면 효과 해석보다 수집·필터 진단이 우선입니다.

**직접 설명:** B의 효과가 좋아 보이더라도 먼저 무엇을 멈춰야 하나요?

## 코드 셀 5 · 원래 배정과 수집 결과 대조

이 사례는 B 사용자 일부를 의도적으로 누락한 예시입니다. 배정 로직이 정확해도 수집·조인·필터에서 편향이 생깁니다.

In [ ]:
broken = load_units("srm")
missing = data[~data.visitor_id.isin(broken.visitor_id)]
display(missing.groupby("variant").size().rename("누락 사용자"))
report = analyze(broken)
print(report["decision"], report["reasons"])
assert report["effect"] is None

**확인과 해석:** B 602명 누락, INVALID, effect is None.

**화면에서 볼 곳:** missing의 버전별 수와 effect=None

**설명 핵심:** 실습은 B 행 절반을 의도적으로 누락했습니다. 실제 서비스에서는 어떤 사용자가 빠졌는지 원본 배정 로그와 대조해야 합니다. 이유를 모른 채 남은 사람만으로 승자를 정하지 않습니다.

**직접 설명:** 누락된 사람은 무작위 일부일까요, 특정 결과를 경험한 사람일까요?

## 코드 셀 6 · 성공 응답만 남기는 함정

실패 사용자를 지우면 관찰 대상이 결과에 의해 바뀝니다. 오류는 분모에 남기고 운영 기준으로도 봅니다.

In [ ]:
all_users = data.groupby("variant").resolved.mean()
only_ok = data[data.error==0].groupby("variant").resolved.mean()
display(pd.DataFrame({"진입 사용자 전체":all_users,"성공 응답만":only_ok}))
display(data.groupby("variant")[["error","feedback_received"]].mean())

**확인과 해석:** 전체 A 55.02% / B 64.04%, 성공 응답만 A 55.57% / B 64.95%.

**화면에서 볼 곳:** 진입 사용자 전체와 성공 응답만의 분모

**설명 핵심:** 오류 사용자를 빼면 실패를 포함해 서비스를 경험한 전체 대상의 결과가 달라집니다. 성공 요청의 응답 품질을 묻는 별도 지표는 가능하지만 원래 목표 지표와 혼용하지 않습니다.

**직접 설명:** ‘답변한 사람 중 만족 비율’과 ‘전체 진입자 중 해결 확인율’의 분모는 어떻게 다른가요?

## 코드 셀 7 · 관찰 지표와 사람의 행동

피드백 미응답은 실제 해결 실패를 뜻하지 않습니다. 수집률과 별도 사용성 관찰을 함께 봅니다.

In [ ]:
observed = data[data.feedback_received==1].groupby("variant").resolved.mean()
display(pd.DataFrame({"전체 사용자 해결 확인율":all_users,"피드백 응답자 내 비율":observed}))
print("응답자만의 만족도와 전체 사용자 지표를 혼동하지 않습니다.")

**확인과 해석:** 전체 A 55.02% / B 64.04%, 피드백 응답자 내 A 62.02% / B 72.26%.

**화면에서 볼 곳:** 전체 사용자와 피드백 응답자의 비율

**설명 핵심:** 응답자가 다르면 숫자가 달라집니다. B의 피드백 버튼만 더 눈에 띄게 바뀌어도 해결 확인율이 달라질 수 있어, 버튼·문구·관찰 창을 일관되게 유지합니다.

**직접 설명:** B에서 피드백 버튼이 더 잘 보이도록 바꾸면 어떤 측정 변화가 생길까요?

## 복습과 다음 단계

MSN의 이미지 카드 실험 사례와 비교해 보세요. 결과에 영향받은 사용자만 필터링되면 판단 방향까지 바뀔 수 있습니다. 다음 Notebook은 유효한 데이터의 불확실성과 운영 기준을 함께 봅니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
